In [1]:
import polars as pl

## UCI Dataset

In [2]:
from gridsight.preprocess.constants import (
    UCI_CLIENTS_TO_DROP,
    UCI_CLIENTS_TO_FILTER,
    UCI_CLIENTS_TO_INTERPOLATE
)
from gridsight.preprocess.explore import get_min_max_timestamps_by_client
from gridsight.preprocess.filter import drop_client_timeseries, filter_client_timeseries
from gridsight.preprocess.impute import interpolate_client_timeseries

from gridsight.validate.constants import UCI_CLIENT_SITES_TO_VALIDATE

In [3]:
FREQUENCY_MINUTES = 15
UCI_DATA_LOAD_PATH = "../data/uci/LD2011_2014.txt"
UCI_DATA_SAVE_PATH = "../data/uci/preprocessed.pq"

In [4]:
# Load data as polars dataframe
UCI_DF = pl.read_csv(
    UCI_DATA_LOAD_PATH,
    has_header=True,
    separator=";",
    decimal_comma=True,
    try_parse_dates=True,
    infer_schema_length=1_000_000
)

# First column should be timestamp column
UCI_DF = UCI_DF.rename({UCI_DF.columns[0]: "timestamp"}).sort(by="timestamp")

In [5]:
# Data processing

for client in UCI_CLIENTS_TO_DROP:
    UCI_DF = drop_client_timeseries(uci_df=UCI_DF, client_name=client)


# Filter client timeseries
for client_name, (start_ts, end_ts) in UCI_CLIENTS_TO_FILTER:
    UCI_DF = filter_client_timeseries(
        uci_df=UCI_DF,
        client_name=client_name,
        start_ts=start_ts,
        end_ts=end_ts
    )


# Interpolate client timeseries
min_max_timestamp_by_client = get_min_max_timestamps_by_client(UCI_DF)
for client_name in UCI_CLIENTS_TO_INTERPOLATE:
    # Get min / max timestamps for this client
    client_min_max_ts = min_max_timestamp_by_client.filter(pl.col("client") == client_name)
    [client_min_ts] = client_min_max_ts["min_timestamp"].to_list()
    [client_max_ts] = client_min_max_ts["max_timestamp"].to_list()

    UCI_DF = interpolate_client_timeseries(
        uci_df=UCI_DF,
        client_name=client_name,
        start_ts=client_min_ts,
        end_ts=client_max_ts,
        interval=f"{FREQUENCY_MINUTES}m"
    )

In [6]:
# Unpivot UCI DF

long_sample_clients_demand_table = (
    UCI_DF
    .unpivot(
        on=[c for c in UCI_DF.columns if c != "timestamp"],
        index="timestamp",
        variable_name="client",
        value_name="demand"
    )
    .join(
        other=min_max_timestamp_by_client,
        on="client",
        how="left",
    )
    .with_columns(in_range=pl.col("timestamp").is_between("min_timestamp", "max_timestamp"),
    )
    .filter(
        pl.col("in_range"),
        pl.col("client").is_in(UCI_CLIENT_SITES_TO_VALIDATE)
    )
    .select(pl.col("timestamp"), pl.col("client"), pl.col("demand"))
)

In [7]:
# Save
long_sample_clients_demand_table.to_pandas().to_parquet(UCI_DATA_SAVE_PATH)